# Learned Risk Score Training

Train a streaming Logistic Regression model for Mempool-TrieGuard risk scoring. This notebook is designed for standard 12.7 GB Colab RAM.


## 1. Mount Drive

Mount Google Drive. The dataset package should be under `MyDrive/mempool_trieguard_colab/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Install Packages

Install Parquet and scikit-learn dependencies. GPU is not required.


In [ ]:
!pip -q install pyarrow pandas numpy scipy scikit-learn


## 3. Configure Paths and Features

Copy Parquet shards from Drive to `/content` before training. This avoids Drive streaming failures such as `Errno 107`.


In [ ]:

from pathlib import Path
import gc, json, pickle, shutil
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler

DRIVE_ROOT = Path('/content/drive/MyDrive/mempool_trieguard_colab')
LOCAL_ROOT = Path('/content/mempool_trieguard_colab')
COPY_DATA_TO_LOCAL = True  # Avoid PyArrow streaming failures on Google Drive.

ROOT = LOCAL_ROOT if COPY_DATA_TO_LOCAL else DRIVE_ROOT
FEATURE_DIR = ROOT / 'features'
OUT_DIR = DRIVE_ROOT / 'trained_models'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = [
    's_addr', 's_type', 's_token', 's_time', 's_value',
    'matched_prefix', 'matched_suffix', 'candidates_scored', 'found_candidate',
    's_addr_x_type', 's_addr_x_token', 's_addr_x_time', 's_addr_x_value',
    'address_gate_weak', 'address_gate_strong',
]
SPLIT_COLUMN = 'split_time'  # Use 'split_victim' for held-out-wallet robustness.
BATCH_SIZE = 50_000
EPOCHS = 2
SEED = 42
THRESHOLD_GRID_SIZE = 1001

def copy_file_if_needed(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        return False
    shutil.copy2(src, dst)
    return True

if COPY_DATA_TO_LOCAL:
    if not DRIVE_ROOT.exists():
        raise RuntimeError(f'Drive root not found: {DRIVE_ROOT}. Mount Drive first.')
    for name in ['manifest.json', 'splits.json', 'feature_schema.json', 'source_hashes.json']:
        src = DRIVE_ROOT / name
        if src.exists():
            copy_file_if_needed(src, LOCAL_ROOT / name)
    drive_feature_dir = DRIVE_ROOT / 'features'
    drive_files = sorted(drive_feature_dir.glob('part-*.parquet'))
    if not drive_files:
        raise RuntimeError(f'No Parquet shards found under {drive_feature_dir}')
    copied = skipped = 0
    for i, src in enumerate(drive_files, start=1):
        changed = copy_file_if_needed(src, FEATURE_DIR / src.name)
        copied += int(changed)
        skipped += int(not changed)
        if i % 16 == 0 or i == len(drive_files):
            print(f'local copy: {i}/{len(drive_files)} files; copied={copied}, skipped={skipped}')

files = sorted(FEATURE_DIR.glob('part-*.parquet'))
print('training root:', ROOT)
print('feature shards:', len(files))
print('first shard:', files[0] if files else None)


## 4. Inspect Dataset Metadata

Check row counts, label counts, split strategy, and feature schema before training.


In [ ]:
manifest = json.loads((ROOT / 'manifest.json').read_text())
splits = json.loads((ROOT / 'splits.json').read_text())
schema = json.loads((ROOT / 'feature_schema.json').read_text())
print(json.dumps({k: manifest[k] for k in ['rows', 'positives', 'negatives', 'shards_exported', 'delay_seconds'] if k in manifest}, indent=2))
print(json.dumps(splits['time'], indent=2))


## 5. Define Low-RAM Helpers

Read Parquet in small batches, build `float32` feature arrays, and evaluate thresholds with streaming histograms.


In [ ]:

def iter_batches(paths, columns, batch_size=BATCH_SIZE):
    for path in paths:
        pf = pq.ParquetFile(path)
        for batch in pf.iter_batches(batch_size=batch_size, columns=columns):
            df = batch.to_pandas(self_destruct=True)
            yield path, df
            del df, batch
            gc.collect()

def split_mask(df, split):
    return df[SPLIT_COLUMN].astype(str).to_numpy() == split

def xy(df):
    return df[FEATURE_COLUMNS].to_numpy(dtype=np.float32, copy=True), df['label'].to_numpy(dtype=np.int8, copy=False)

def prf(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

def metrics_from_counts(threshold, tp, fp, fn, tn):
    precision, recall, f1 = prf(tp, fp, fn)
    return {
        'threshold': float(threshold),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn),
        'rows': int(tp + fp + fn + tn),
    }

def score_histogram_for_split(split, scorer, grid_size=THRESHOLD_GRID_SIZE):
    thresholds = np.linspace(0.0, 1.0, grid_size, dtype=np.float64)
    pos_hist = np.zeros(grid_size, dtype=np.int64)
    neg_hist = np.zeros(grid_size, dtype=np.int64)
    rows = positives = 0
    for path, df in iter_batches(files, cols):
        mask = split_mask(df, split)
        if not mask.any():
            continue
        X, y = xy(df.loc[mask])
        scores = np.clip(np.asarray(scorer(X), dtype=np.float64), 0.0, 1.0)
        idx = np.searchsorted(thresholds, scores, side='right') - 1
        idx = np.clip(idx, 0, grid_size - 1)
        pos_hist += np.bincount(idx[y == 1], minlength=grid_size)
        neg_hist += np.bincount(idx[y == 0], minlength=grid_size)
        rows += int(y.size)
        positives += int(y.sum())
        del X, y, scores, idx, df
        gc.collect()
    return thresholds, pos_hist, neg_hist, {'rows': rows, 'positives': positives, 'negatives': rows - positives}

def best_threshold_from_hist(thresholds, pos_hist, neg_hist):
    total_pos = int(pos_hist.sum())
    total_neg = int(neg_hist.sum())
    pred_pos_pos = np.cumsum(pos_hist[::-1])[::-1]
    pred_pos_neg = np.cumsum(neg_hist[::-1])[::-1]
    best = None
    for i, threshold in enumerate(thresholds):
        tp = int(pred_pos_pos[i])
        fp = int(pred_pos_neg[i])
        fn = total_pos - tp
        tn = total_neg - fp
        item = metrics_from_counts(float(threshold), tp, fp, fn, tn)
        key = (item['f1'], item['precision'], item['recall'], -item['threshold'])
        if best is None or key > (best['f1'], best['precision'], best['recall'], -best['threshold']):
            best = item
    return best

def metrics_at_threshold_from_hist(thresholds, pos_hist, neg_hist, threshold):
    idx = int(np.searchsorted(thresholds, threshold, side='left'))
    idx = min(max(idx, 0), len(thresholds) - 1)
    pred_pos_pos = np.cumsum(pos_hist[::-1])[::-1]
    pred_pos_neg = np.cumsum(neg_hist[::-1])[::-1]
    tp = int(pred_pos_pos[idx])
    fp = int(pred_pos_neg[idx])
    fn = int(pos_hist.sum()) - tp
    tn = int(neg_hist.sum()) - fp
    return metrics_from_counts(float(thresholds[idx]), tp, fp, fn, tn)


## 6. Smoke Test One Shard

Verify that the first shard can be read and contains the expected split and label columns.


In [ ]:

# Read one shard before full training.
sample = pd.read_parquet(files[0], columns=['label', SPLIT_COLUMN] + FEATURE_COLUMNS)
print(sample.head())
print(sample[SPLIT_COLUMN].value_counts().to_dict())
print(sample['label'].value_counts().to_dict())
del sample
gc.collect()


## 7. Fit Scaler

Fit `StandardScaler` on the train split only. This is a streaming pass over the dataset.


In [ ]:

# Pass 1: fit feature normalization on train only.
cols = FEATURE_COLUMNS + ['label', SPLIT_COLUMN]
scaler = StandardScaler()
train_rows = 0
train_counts = {0: 0, 1: 0}

for path, df in iter_batches(files, cols):
    mask = split_mask(df, 'train')
    if not mask.any():
        continue
    X, y = xy(df.loc[mask])
    scaler.partial_fit(X)
    train_rows += int(y.size)
    train_counts[0] += int((y == 0).sum())
    train_counts[1] += int((y == 1).sum())
    del X, y, df
    gc.collect()

print('train rows:', train_rows)
print('train label counts:', train_counts)
if train_counts[0] == 0 or train_counts[1] == 0:
    raise RuntimeError(f'train split must contain both classes: {train_counts}')


## 8. Train Logistic Regression

Train `SGDClassifier(loss="log_loss")` with class-balanced sample weights. This is the learned risk score.


In [ ]:

# Pass 2: train streaming Logistic Regression.
classes = np.array([0, 1], dtype=np.int8)
total = train_counts[0] + train_counts[1]
class_weights = {
    0: total / (2 * train_counts[0]),
    1: total / (2 * train_counts[1]),
}
clf = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    alpha=1e-5,
    learning_rate='optimal',
    random_state=SEED,
    average=True,
)

for epoch in range(EPOCHS):
    seen = 0
    for path, df in iter_batches(files, cols):
        mask = split_mask(df, 'train')
        if not mask.any():
            continue
        X, y = xy(df.loc[mask])
        sample_weight = np.where(y == 1, class_weights[1], class_weights[0])
        clf.partial_fit(scaler.transform(X), y, classes=classes, sample_weight=sample_weight)
        seen += int(y.size)
        del X, y, sample_weight, df
        gc.collect()
    print(f'epoch {epoch + 1}/{EPOCHS}: rows={seen}')


## 9. Select Threshold and Evaluate

Choose the alert threshold on validation only, freeze it, then report final metrics on test.


In [ ]:

def logistic_scores(X):
    return clf.predict_proba(scaler.transform(X))[:, 1]

val_thresholds, val_pos_hist, val_neg_hist, val_summary = score_histogram_for_split('val', logistic_scores)
best = best_threshold_from_hist(val_thresholds, val_pos_hist, val_neg_hist)

test_thresholds, test_pos_hist, test_neg_hist, test_summary = score_histogram_for_split('test', logistic_scores)
test_metrics = metrics_at_threshold_from_hist(test_thresholds, test_pos_hist, test_neg_hist, best['threshold'])

print('validation rows:', val_summary)
print('test rows:', test_summary)
print('validation threshold:', best)
print('test metrics:', test_metrics)


## 10. Compare Context-Modifier Scores

Evaluate address-only, manual full score, no-time, and gated context modifiers on the same validation/test splits.


In [ ]:
FEATURE_INDEX = {name: i for i, name in enumerate(FEATURE_COLUMNS)}
MANUAL_WEIGHTS = np.asarray([0.30, 0.20, 0.20, 0.15, 0.15], dtype=np.float32)
NO_TIME_WEIGHTS = np.asarray([0.30, 0.20, 0.20, 0.0, 0.15], dtype=np.float32)
NO_TIME_WEIGHTS = NO_TIME_WEIGHTS / NO_TIME_WEIGHTS.sum()

CONTEXT_SCORE_SPECS = [
    {'name': 'ctx_boost_ttv_weak', 'kind': 'boost', 'type': 0.05, 'token': 0.05, 'time': 0.0, 'value': 0.025},
    {'name': 'ctx_boost_ttv_mid', 'kind': 'boost', 'type': 0.10, 'token': 0.10, 'time': 0.0, 'value': 0.05},
    {'name': 'ctx_boost_ttv_strong', 'kind': 'boost', 'type': 0.20, 'token': 0.20, 'time': 0.0, 'value': 0.10},
    {'name': 'ctx_boost_tttv_weak', 'kind': 'boost', 'type': 0.05, 'token': 0.05, 'time': 0.025, 'value': 0.025},
    {'name': 'ctx_boost_type_only', 'kind': 'boost', 'type': 0.15, 'token': 0.0, 'time': 0.0, 'value': 0.0},
    {'name': 'ctx_boost_token_only', 'kind': 'boost', 'type': 0.0, 'token': 0.15, 'time': 0.0, 'value': 0.0},
    {'name': 'ctx_mult_ttv_weak', 'kind': 'multiplicative', 'type': 0.10, 'token': 0.10, 'time': 0.0, 'value': 0.05},
    {'name': 'ctx_mult_ttv_mid', 'kind': 'multiplicative', 'type': 0.20, 'token': 0.20, 'time': 0.0, 'value': 0.10},
    {'name': 'ctx_mult_tttv_weak', 'kind': 'multiplicative', 'type': 0.10, 'token': 0.10, 'time': 0.05, 'value': 0.05},
    {'name': 'ctx_gate_ttv_90', 'kind': 'gate', 'base': 0.90, 'type': 0.4, 'token': 0.4, 'time': 0.0, 'value': 0.2},
    {'name': 'ctx_gate_ttv_85', 'kind': 'gate', 'base': 0.85, 'type': 0.4, 'token': 0.4, 'time': 0.0, 'value': 0.2},
    {'name': 'ctx_gate_ttv_80', 'kind': 'gate', 'base': 0.80, 'type': 0.4, 'token': 0.4, 'time': 0.0, 'value': 0.2},
    {'name': 'ctx_gate_ttv_75', 'kind': 'gate', 'base': 0.75, 'type': 0.4, 'token': 0.4, 'time': 0.0, 'value': 0.2},
    {'name': 'ctx_gate_type_token_85', 'kind': 'gate', 'base': 0.85, 'type': 0.5, 'token': 0.5, 'time': 0.0, 'value': 0.0},
    {'name': 'ctx_gate_token_85', 'kind': 'gate', 'base': 0.85, 'type': 0.0, 'token': 1.0, 'time': 0.0, 'value': 0.0},
    {'name': 'ctx_gate_balanced_tuned_30', 'kind': 'gate', 'addr': 'balanced', 'base': 0.30, 'type': 0.20, 'token': 0.55, 'time': 0.0, 'value': 0.0},
    {'name': 'ctx_gate_balanced_tuned_30_with_value', 'kind': 'gate', 'addr': 'balanced', 'base': 0.30, 'type': 0.20, 'token': 0.55, 'time': 0.0, 'value': 0.05},
    {'name': 'ctx_gate_balanced_tuned_30_with_time', 'kind': 'gate', 'addr': 'balanced', 'base': 0.30, 'type': 0.20, 'token': 0.55, 'time': 0.20, 'value': 0.05},
    {'name': 'ctx_gate_tuned_30_no_type', 'kind': 'gate', 'base': 0.30, 'type': 0.0, 'token': 0.55, 'time': 0.20, 'value': 0.05},
    {'name': 'ctx_gate_tuned_30_no_token', 'kind': 'gate', 'base': 0.30, 'type': 0.20, 'token': 0.0, 'time': 0.20, 'value': 0.05},
    {'name': 'ctx_gate_tuned_30_no_time', 'kind': 'gate', 'base': 0.30, 'type': 0.20, 'token': 0.55, 'time': 0.0, 'value': 0.05},
    {'name': 'ctx_gate_tuned_30_no_value', 'kind': 'gate', 'base': 0.30, 'type': 0.20, 'token': 0.55, 'time': 0.20, 'value': 0.0},
    {'name': 'ctx_gate_temporal_80', 'kind': 'gate', 'base': 0.80, 'type': 0.35, 'token': 0.35, 'time': 0.15, 'value': 0.15},
    {'name': 'ctx_gate_temporal_no_type', 'kind': 'gate', 'base': 0.80, 'type': 0.0, 'token': 0.35, 'time': 0.15, 'value': 0.15},
    {'name': 'ctx_gate_temporal_no_token', 'kind': 'gate', 'base': 0.80, 'type': 0.35, 'token': 0.0, 'time': 0.15, 'value': 0.15},
    {'name': 'ctx_gate_temporal_no_time', 'kind': 'gate', 'base': 0.80, 'type': 0.35, 'token': 0.35, 'time': 0.0, 'value': 0.15},
    {'name': 'ctx_gate_temporal_no_value', 'kind': 'gate', 'base': 0.80, 'type': 0.35, 'token': 0.35, 'time': 0.15, 'value': 0.0},
]

def manual_full_scores(X):
    return X[:, [FEATURE_INDEX['s_addr'], FEATURE_INDEX['s_type'], FEATURE_INDEX['s_token'], FEATURE_INDEX['s_time'], FEATURE_INDEX['s_value']]] @ MANUAL_WEIGHTS

def no_time_scores(X):
    return X[:, [FEATURE_INDEX['s_addr'], FEATURE_INDEX['s_type'], FEATURE_INDEX['s_token'], FEATURE_INDEX['s_time'], FEATURE_INDEX['s_value']]] @ NO_TIME_WEIGHTS

def address_only_scores(X):
    return X[:, FEATURE_INDEX['s_addr']]

def context_modifier_scores(X, spec):
    s_addr = X[:, FEATURE_INDEX['s_addr']]
    modifier = (
        spec['type'] * X[:, FEATURE_INDEX['s_type']]
        + spec['token'] * X[:, FEATURE_INDEX['s_token']]
        + spec['time'] * X[:, FEATURE_INDEX['s_time']]
        + spec['value'] * X[:, FEATURE_INDEX['s_value']]
    )
    if spec['kind'] == 'multiplicative':
        scores = s_addr * (1.0 + modifier)
    elif spec['kind'] == 'gate':
        total_weight = spec['type'] + spec['token'] + spec['time'] + spec['value']
        conditioned_time = X[:, FEATURE_INDEX['s_time']] * np.maximum(X[:, FEATURE_INDEX['s_type']], X[:, FEATURE_INDEX['s_token']])
        gated_modifier = (
            spec['type'] * X[:, FEATURE_INDEX['s_type']]
            + spec['token'] * X[:, FEATURE_INDEX['s_token']]
            + spec['time'] * conditioned_time
            + spec['value'] * X[:, FEATURE_INDEX['s_value']]
        )
        context = gated_modifier / total_weight if total_weight > 0 else np.zeros_like(s_addr)
        base = spec['base']
        scores = s_addr * (base + (1.0 - base) * context)
    else:
        # Context only changes ranking after the address gate has fired.
        scores = s_addr + (s_addr * modifier)
    return np.clip(scores, 0.0, 1.0)

CONTEXT_SCORERS = [
    ('address_only', address_only_scores),
    ('manual_full_score', manual_full_scores),
    ('manual_no_time', no_time_scores),
]
CONTEXT_SCORERS += [(spec['name'], lambda X, spec=spec: context_modifier_scores(X, spec)) for spec in CONTEXT_SCORE_SPECS]

def score_histograms_for_split_multi(split, scorers, grid_size=THRESHOLD_GRID_SIZE):
    thresholds = np.linspace(0.0, 1.0, grid_size, dtype=np.float64)
    hists = {
        name: {
            'pos': np.zeros(grid_size, dtype=np.int64),
            'neg': np.zeros(grid_size, dtype=np.int64),
        }
        for name, _ in scorers
    }
    rows = positives = 0
    for path, df in iter_batches(files, cols):
        mask = split_mask(df, split)
        if not mask.any():
            continue
        X, y = xy(df.loc[mask])
        rows += int(y.size)
        positives += int(y.sum())
        for name, scorer in scorers:
            scores = np.clip(np.asarray(scorer(X), dtype=np.float64), 0.0, 1.0)
            idx = np.searchsorted(thresholds, scores, side='right') - 1
            idx = np.clip(idx, 0, grid_size - 1)
            hists[name]['pos'] += np.bincount(idx[y == 1], minlength=grid_size)
            hists[name]['neg'] += np.bincount(idx[y == 0], minlength=grid_size)
            del scores, idx
        del X, y, df
        gc.collect()
    summary = {'rows': rows, 'positives': positives, 'negatives': rows - positives}
    return thresholds, hists, summary

val_ctx_thresholds, val_ctx_hists, val_ctx_summary = score_histograms_for_split_multi('val', CONTEXT_SCORERS)
test_ctx_thresholds, test_ctx_hists, test_ctx_summary = score_histograms_for_split_multi('test', CONTEXT_SCORERS)

FIXED_THRESHOLD = 0.40
context_results = []
context_fixed_results = []
for name, _ in CONTEXT_SCORERS:
    val_best = best_threshold_from_hist(val_ctx_thresholds, val_ctx_hists[name]['pos'], val_ctx_hists[name]['neg'])
    test_at_val = metrics_at_threshold_from_hist(test_ctx_thresholds, test_ctx_hists[name]['pos'], test_ctx_hists[name]['neg'], val_best['threshold'])
    fixed_val = metrics_at_threshold_from_hist(val_ctx_thresholds, val_ctx_hists[name]['pos'], val_ctx_hists[name]['neg'], FIXED_THRESHOLD)
    fixed_test = metrics_at_threshold_from_hist(test_ctx_thresholds, test_ctx_hists[name]['pos'], test_ctx_hists[name]['neg'], FIXED_THRESHOLD)
    context_results.append({'name': name, 'validation': val_best, 'test': test_at_val})
    context_fixed_results.append({'name': name, 'validation': fixed_val, 'test': fixed_test})

context_results = sorted(
    context_results,
    key=lambda item: (item['validation']['f1'], item['validation']['precision'], item['validation']['recall']),
    reverse=True,
)
context_fixed_results = sorted(
    context_fixed_results,
    key=lambda item: (item['test']['f1'], item['test']['precision'], item['test']['recall']),
    reverse=True,
)
best_context_by_validation = context_results[0]
best_context_fixed_threshold = context_fixed_results[0]

print('context validation rows:', val_ctx_summary)
print('context test rows:', test_ctx_summary)
print('best context score by validation:', json.dumps(best_context_by_validation, indent=2))
print('top context scores:', json.dumps(context_results[:6], indent=2))
print('best context score at fixed tau:', json.dumps(best_context_fixed_threshold, indent=2))
print('top fixed-tau context scores:', json.dumps(context_fixed_results[:6], indent=2))


## 11. Candidate Coverage Diagnostic

Count positives that have no retrieved candidate. Candidate-level risk scores cannot recover these rows.


In [ ]:
def candidate_coverage(split):
    counts = {
        'rows': 0,
        'positives': 0,
        'negatives': 0,
        'found_candidate_rows': 0,
        'no_candidate_rows': 0,
        'positive_found_candidate': 0,
        'positive_no_candidate': 0,
        'negative_found_candidate': 0,
        'negative_no_candidate': 0,
    }
    for path, df in iter_batches(files, ['label', 'found_candidate', SPLIT_COLUMN]):
        mask = split_mask(df, split)
        if not mask.any():
            continue
        labels = df.loc[mask, 'label'].to_numpy(dtype=np.int8, copy=False)
        found = df.loc[mask, 'found_candidate'].to_numpy(dtype=np.int8, copy=False).astype(bool)
        positive = labels == 1
        negative = ~positive
        counts['rows'] += int(labels.size)
        counts['positives'] += int(labels.sum())
        counts['negatives'] += int(labels.size - labels.sum())
        counts['found_candidate_rows'] += int(found.sum())
        counts['no_candidate_rows'] += int(labels.size - found.sum())
        counts['positive_found_candidate'] += int(np.logical_and(positive, found).sum())
        counts['positive_no_candidate'] += int(np.logical_and(positive, ~found).sum())
        counts['negative_found_candidate'] += int(np.logical_and(negative, found).sum())
        counts['negative_no_candidate'] += int(np.logical_and(negative, ~found).sum())
        del labels, found, positive, negative, df
        gc.collect()
    return counts

candidate_coverage_summary = {
    'val': candidate_coverage('val'),
    'test': candidate_coverage('test'),
}
print(json.dumps(candidate_coverage_summary, indent=2))


## 12. Save Outputs

Save metrics, the LR model, context sweep, and candidate coverage.


In [ ]:

metrics = {
    'model': 'logistic_sgd',
    'split_column': SPLIT_COLUMN,
    'feature_columns': FEATURE_COLUMNS,
    'batch_size': BATCH_SIZE,
    'threshold_grid_size': THRESHOLD_GRID_SIZE,
    'validation_summary': val_summary,
    'test_summary': test_summary,
    'validation': best,
    'test': test_metrics,
    'context_modifier_sweep': context_results,
    'context_modifier_fixed_threshold': context_fixed_results,
    'fixed_threshold': FIXED_THRESHOLD,
    'best_context_by_validation': best_context_by_validation,
    'candidate_coverage': candidate_coverage_summary,
}
(OUT_DIR / 'logistic_sgd_metrics.json').write_text(json.dumps(metrics, indent=2))
with (OUT_DIR / 'logistic_sgd_model.pkl').open('wb') as f:
    pickle.dump({'model': clf, 'scaler': scaler, 'feature_columns': FEATURE_COLUMNS, 'split_column': SPLIT_COLUMN}, f)
print('saved to', OUT_DIR)


## Optional Random Forest

Run this only on high-RAM Colab or a local machine.


In [ ]:

# Optional high-RAM only. Keep commented on standard Colab.
# from sklearn.ensemble import RandomForestClassifier
# RF_MAX_SAMPLES = 300_000
# rng = np.random.default_rng(SEED)
# pos_parts, neg_parts = [], []
# for path, df in iter_batches(files, cols):
#     mask = split_mask(df, 'train')
#     if not mask.any():
#         continue
#     X, y = xy(df.loc[mask])
#     pos_idx = np.flatnonzero(y == 1)
#     neg_idx = np.flatnonzero(y == 0)
#     if len(pos_idx): pos_parts.append(X[rng.choice(pos_idx, size=min(len(pos_idx), 800), replace=False)])
#     if len(neg_idx): neg_parts.append(X[rng.choice(neg_idx, size=min(len(neg_idx), 800), replace=False)])
#     del X, y, df
#     gc.collect()
# X_rf = np.concatenate(pos_parts + neg_parts, axis=0)
# y_rf = np.concatenate([np.ones(sum(len(x) for x in pos_parts), dtype=np.int8), np.zeros(sum(len(x) for x in neg_parts), dtype=np.int8)])
# order = rng.permutation(len(y_rf))[:RF_MAX_SAMPLES]
# rf = RandomForestClassifier(n_estimators=100, max_depth=16, min_samples_leaf=50, class_weight='balanced_subsample', n_jobs=-1, random_state=SEED)
# rf.fit(X_rf[order], y_rf[order])
